# Motor-Imagery EEG — LOSO benchmark on a Kaggle GPU

Runs the **Leave-One-Subject-Out** benchmark (the subject-independent, honest number) for all 5 models, since Colab's GPU quota ran out. LOSO is the most interesting test for the Denoising-EEGNet — cross-subject noise is exactly what its front-end targets.

### Before you run — two toggles in the right-hand panel
1. **Settings ▸ Accelerator ▸ GPU** (T4 or P100).
2. **Settings ▸ Internet ▸ On** — required to `git clone`, `pip install`, and download EEGMMIDB.

### How to get this notebook into Kaggle
On kaggle.com: **Create ▸ New Notebook ▸ File ▸ Import Notebook ▸ upload** this `.ipynb` (download it from the repo first), or import from the GitHub URL.

### Tip for the long run
LOSO retrains on N−1 subjects every fold, so it's heavier than within-subject (~2–3 h for 40 subjects). Use **Save Version ▸ Save & Run All (Commit)** to run it headless — you don't need to keep the tab open, and the outputs are saved to the version.

## 1. Confirm GPU + internet

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU — enable Settings > Accelerator > GPU'
import urllib.request
try:
    urllib.request.urlopen('https://physionet.org', timeout=10); print('internet: OK')
except Exception as e:
    print('internet: OFF — enable Settings > Internet >', e)

## 2. Clone the repo (into /kaggle/working)

In [ ]:
import os
os.chdir('/kaggle/working')
REPO = 'https://github.com/ol1sa/Motor-Imagery-EEG-.git'
if not os.path.isdir('Motor-Imagery-EEG-'):
    !git clone -q $REPO
os.chdir('/kaggle/working/Motor-Imagery-EEG-')
!git pull -q
print('cwd:', os.getcwd())

## 3. Install only what Kaggle lacks
Kaggle ships GPU PyTorch + the scientific stack. We add MNE, pyRiemann and statsmodels on top and **do not** touch torch/NumPy (the repo's CPU-pinned `requirements.txt` would break the GPU build).

In [ ]:
!pip install -q mne pyriemann statsmodels pooch
import torch, mne, pyriemann
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('mne', mne.__version__, '| pyriemann', pyriemann.__version__)

## 4. Configure (same 40-subject cohort as the within-subject table)
Keeping the cohort identical makes the LOSO numbers directly comparable to the within-subject ones already in the README.

In [ ]:
import yaml

N_SUBJECTS = 40       # match the within-subject run; lower it if you want a quicker pass
MODELS = ['csp_lda', 'csp_svm', 'riemann_lr', 'eegnet', 'denoising_eegnet']

cfg = yaml.safe_load(open('configs/binary.yaml'))
cfg['name'] = 'kaggle'
cfg['data']['subjects'] = list(range(1, N_SUBJECTS + 1))
cfg['model']['names'] = MODELS
yaml.safe_dump(cfg, open('configs/kaggle.yaml', 'w'), sort_keys=False)
print(f'config: {N_SUBJECTS} subjects, LOSO, models = {MODELS}')

## 5. Run LOSO
Downloads + preprocesses (CPU, cached after first pass) then trains; the deep models use the GPU automatically. `-u` keeps the log live.

In [ ]:
!PYTHONPATH=src python -u -m mibci.run --config configs/kaggle.yaml --experiment binary --cv loso

## 6. Show the LOSO results

In [ ]:
import pandas as pd, glob
from IPython.display import Image, display
tag = 'kaggle_binary_loso'
display(pd.read_csv(f'artifacts/summary_{tag}.csv'))
display(pd.read_csv(f'artifacts/stats_{tag}.csv'))
for png in sorted(glob.glob(f'artifacts/*{tag}*.png')):
    print(png); display(Image(png))

## 7. Save the results so you can download them
Anything copied into `/kaggle/working` is saved as the notebook's **Output** (Data tab / version output) and downloadable. We also zip it for convenience.

In [ ]:
import shutil, glob, os
os.makedirs('/kaggle/working/results', exist_ok=True)
for f in glob.glob('artifacts/*kaggle_binary_loso*'):
    shutil.copy(f, '/kaggle/working/results/')
shutil.make_archive('/kaggle/working/mibci_kaggle_loso_results', 'zip', '/kaggle/working/results')
print('saved to /kaggle/working/results/ and mibci_kaggle_loso_results.zip')
print(os.listdir('/kaggle/working/results'))

In [ ]:
# (optional) push results straight back to GitHub instead of downloading.
# 1) Add your PAT as a Kaggle Secret named GITHUB_TOKEN (Add-ons > Secrets),
#    or paste inline (and clear the cell after).
#
# from kaggle_secrets import UserSecretsClient
# TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
# !git config user.name 'ol1sa' && git config user.email 'olisaogbue@icloud.com'
# !git add -f artifacts/*kaggle_binary_loso*
# !git commit -q -m 'Add Kaggle LOSO benchmark results (5 models)'
# !git push -q https://$TOKEN@github.com/ol1sa/Motor-Imagery-EEG-.git main
# print('pushed')